In [ ]:
import numpy as np
import os
import os.path as op
import pandas as pd
import time
import yaml
import argparse


from alternet.splicefactor_evidence import compute_dominance_metrics, calculate_transcript_usage, tf_sf_disambigouation_fully_as_aware, compute_set_c
from alternet.compare_nets import *
from alternet.annotation import *
from alternet.data_preprocessing import *




def write_dict_to_yaml(data, filepath):
    """Write dictionary to YAML file."""
    with open(filepath, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)

results_path = '/data/bionets/og86asub/alternet-project/alternet/alternet_results_v3'
experiment_name = 'Bladder'



os.makedirs(results_path, exist_ok = True)

regulator_file = '/data/bionets/og86asub/alternet-project/alternet/data/regulator_db.tsv'
regulator_list  = pd.read_csv(regulator_file, sep = '\t')

biomart_file = '/data/bionets/og86asub/alternet-project/alternet/data/biomart_biotype.txt'
biomart = pd.read_csv(biomart_file, sep='\t')
tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2tx = biomart.groupby('Gene stable ID')['Transcript stable ID'].apply(set).to_dict()
gene2regtype = dict(zip(regulator_list['Transcript stable ID'], regulator_list['Regulator_type'])) | dict(zip(regulator_list['Gene stable ID'], regulator_list['Regulator_type']))

appris = '/data/bionets/og86asub/alternet-project/alternet/data/appris_data.appris.txt'
digger = '/data/bionets/og86asub/alternet-project/alternet/data/digger_data.csv'

appris_df = pd.read_csv(appris, sep='\t')
digger_df = pd.read_csv(digger, low_memory=False)

tf_database = create_transcipt_annotation_database(
    tf_list=regulator_list, appris_df=appris_df, digger=digger_df
)

transcript_file = '/data/bionets/og86asub/alternet-project/data/gtex/Bladder.tsv'
transcript_data = pd.read_csv(transcript_file, sep = '\t', index_col = 0)
sample_cols = [c for c in transcript_data.columns if c not in ['transcript_id', 'gene_id']]

source_as = '/data/bionets/og86asub/alternet-project/results_v3/results_v3/Bladder/source_as.tsv'




                   source           target  frequency  mean_importance  \
0         ENST00000000442  ENSG00000000003          9         0.181330   
1         ENST00000000442  ENSG00000000005         10         0.472867   
2         ENST00000000442  ENSG00000000419          2         0.080064   
3         ENST00000000442  ENSG00000000457          6         0.018363   
4         ENST00000000442  ENSG00000000460          1         0.089862   
...                   ...              ...        ...              ...   
32915533  ENST00000640765  ENSG00000284292          1         0.234024   
32915534  ENST00000640765  ENSG00000284308          3         0.083732   
32915535  ENST00000640765  ENSG00000284505          7         0.003370   
32915536  ENST00000640765  ENSG00000284512          1         0.012894   
32915537  ENST00000640765  ENSG00000284526          5         0.046013   

          median_importance  
0                  0.000126  
1                  0.380954  
2                  0.

In [7]:
as_source_grn = pd.read_csv(source_as, sep='\t')
print(as_source_grn)
as_source_grn = canonical_names(as_source_grn, tx2gene, gene2regtype)
as_source_grn = filter_edges(as_source_grn, frequency=10, imp_col = 'median_importance')

fully_as = '/data/bionets/og86asub/alternet-project/results_v3/results_v3/Bladder/fully_as.tsv'
fully_as_aware = pd.read_csv(fully_as, sep='\t')
fully_as_aware = canonical_names(fully_as_aware, tx2gene, gene2regtype)
fully_as_aware = filter_edges(fully_as_aware,  frequency=10, imp_col = 'median_importance')

canonical = '/data/bionets/og86asub/alternet-project/results_v3/results_v3/Bladder/canonical.tsv'

canonical_grn = pd.read_csv(canonical, sep='\t')
canonical_grn = canonical_names(canonical_grn, tx2gene, gene2regtype)
canonical_grn = filter_edges(canonical_grn, frequency=10, imp_col = 'median_importance')

print(canonical_grn)

networks = pd.concat([canonical_grn, as_source_grn, fully_as_aware])
networks = get_best_variable(networks)

                   source           target  frequency  mean_importance  \
0         ENST00000000442  ENSG00000000003          9         0.181330   
1         ENST00000000442  ENSG00000000005         10         0.472867   
2         ENST00000000442  ENSG00000000419          2         0.080064   
3         ENST00000000442  ENSG00000000457          6         0.018363   
4         ENST00000000442  ENSG00000000460          1         0.089862   
...                   ...              ...        ...              ...   
32915533  ENST00000640765  ENSG00000284292          1         0.234024   
32915534  ENST00000640765  ENSG00000284308          3         0.083732   
32915535  ENST00000640765  ENSG00000284505          7         0.003370   
32915536  ENST00000640765  ENSG00000284512          1         0.012894   
32915537  ENST00000640765  ENSG00000284526          5         0.046013   

          median_importance  
0                  0.000126  
1                  0.380954  
2                  0.

In [ ]:



gene_dominance, gene_n_isoforms, tx_expression_share  = compute_dominance_metrics(transcript_data, sample_cols)
networks = plausibility_filtering(networks, gene_dominance, r_dom = 0.9)
print(networks)

usage_df, reliability_df = calculate_transcript_usage(transcript_data)
transcript_data_temp = transcript_data.set_index('transcript_id')[sample_cols]
usage_df = usage_df.set_index('transcript_id')[sample_cols]

# Source == TF net
tf_net = networks[(networks.reg_type == 'TF')].copy()




                   source           target  frequency  mean_importance  \
0         ENSG00000001167  ENSG00000000003         10        10.545621   
8         ENSG00000001167  ENSG00000001084         10         1.317990   
10        ENSG00000001167  ENSG00000001461         10         4.740594   
12        ENSG00000001167  ENSG00000001617         10         2.165028   
19        ENSG00000001167  ENSG00000002587         10         6.589235   
...                   ...              ...        ...              ...   
68999559  ENST00000640765  ENST00000604804         10         2.815125   
68999571  ENST00000640765  ENST00000605552         10         1.876706   
68999724  ENST00000640765  ENST00000611969         10         0.989064   
68999805  ENST00000640765  ENST00000613958         10         0.665655   
69000371  ENST00000640765  ENST00000636714         10         1.471079   

          median_importance source_type target_type      source_gene  \
0                 10.967363        gene

KeyError: 'importance'

In [11]:
# source == SF net
sf_candidates = networks[(networks.reg_type == 'SF') & (networks.target_type=='transcript') & (networks.source_type=='transcript')]
if sf_candidates.shape[0]>0:
    sf_net = compute_set_c(sf_candidates, transcript_data, gene2tx, usage_df, reliability_df, sample_cols, epsilon=1e-6, n_cores=16)
else:
    sf_net = None




No corrletation
No corrletation
No corrletation
No corrletation
1000
No corrletation
No corrletation
No corrletation
No corrletation
No corrletation
No corrletation
2000
No corrletation
No corrletation
No corrletation
3000
No corrletation
No corrletation
No corrletation
No corrletation
4000
No corrletation
No corrletation
No corrletation
5000
No corrletation
6000
No corrletation
No corrletation
7000
No corrletation
No corrletation
No corrletation
No corrletation
No corrletation
No corrletation
No corrletation
No corrletation
8000
No corrletation
No corrletation
9000
No corrletation
No corrletation
No corrletation
10000
No corrletation
No corrletation
No corrletation
No corrletation
No corrletation
11000
No corrletation
No corrletation
No corrletation
No corrletation
12000
No corrletation
No corrletation
No corrletation
13000
No corrletation
No corrletation
No corrletation
14000
No corrletation
No corrletation
15000
No corrletation
No corrletation
16000
No corrletation
No corrletation
N

KeyError: 'importance'

In [9]:
sf_candidates

,source,target,frequency,mean_importance,median_importance,source_type,target_type,source_gene,target_gene,source_transcript,...,reg_type,edge_gg,edge_type,is_unique_edge,importance_ratio,category,n_equivalent_edge_types,reg_dominance,target_dominance,is_plausible
202666,ENST00000156109,ENST00000011619,10,7.172992,6.754724,transcript,transcript,ENSG00000068394,ENSG00000010017,ENST00000156109,...,SF,ENSG00000068394_ENSG00000010017,transcript-transcript,True,1.000001,specific_strict,0,1.000000,1.000000,False
202891,ENST00000156109,ENST00000216038,10,9.429034,7.231547,transcript,transcript,ENSG00000068394,ENSG00000100220,ENST00000156109,...,SF,ENSG00000068394_ENSG00000100220,transcript-transcript,True,1.000001,specific_strict,0,1.000000,1.000000,False
202928,ENST00000156109,ENST00000216378,10,1.659159,0.752891,transcript,transcript,ENSG00000068394,ENSG00000100490,ENST00000156109,...,SF,ENSG00000068394_ENSG00000100490,transcript-transcript,True,1.000001,specific_strict,0,1.000000,0.821428,True
203129,ENST00000156109,ENST00000223190,10,6.215969,6.573494,transcript,transcript,ENSG00000068394,ENSG00000106459,ENST00000156109,...,SF,ENSG00000068394_ENSG00000106459,transcript-transcript,True,1.000001,specific_strict,0,1.000000,0.873286,True
203265,ENST00000156109,ENST00000230124,10,3.659100,2.735812,transcript,transcript,ENSG00000068394,ENSG00000112367,ENST00000156109,...,SF,ENSG00000068394_ENSG00000112367,transcript-transcript,True,1.000001,specific_strict,0,1.000000,0.949730,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68979390,ENST00000640218,ENST00000617231,10,11.195415,11.691152,transcript,transcript,ENSG00000153187,ENSG00000161847,ENST00000640218,...,SF,ENSG00000153187_ENSG00000161847,transcript-transcript,True,1.000001,specific_strict,0,0.329376,1.000000,True
68979714,ENST00000640218,ENST00000633942,10,2.680658,0.935525,transcript,transcript,ENSG00000153187,ENSG00000167676,ENST00000640218,...,SF,ENSG00000153187_ENSG00000167676,transcript-transcript,True,1.000001,specific_strict,0,0.329376,0.741499,True
68979730,ENST00000640218,ENST00000634739,10,7.821435,6.679161,transcript,transcript,ENSG00000153187,ENSG00000058272,ENST00000640218,...,SF,ENSG00000153187_ENSG00000058272,transcript-transcript,True,1.000001,specific_strict,0,0.329376,0.423384,True
68979791,ENST00000640218,ENST00000637174,10,4.431533,1.792362,transcript,transcript,ENSG00000153187,ENSG00000198825,ENST00000640218,...,SF,ENSG00000153187_ENSG00000198825,transcript-transcript,False,0.710045,other,0,0.329376,0.913166,True


In [ ]:

# Decide if TF_SF is splice factor or transcription factor
if networks[(networks.reg_type == 'TF_SF')].shape[0]>0:
    nn = tf_sf_disambigouation_fully_as_aware(networks[(networks.reg_type == 'TF_SF') & (networks.target_type=='transcript')], regulator_list, transcript_data_temp, usage_df, reliability_df,
                                            RHO_MIN = 0.3,  Q_MIN = 0.05, DU_MIN = 0.1, n_cores = 16)

    ab = compute_set_c(nn[nn.tfsf_category == 'tfsf_sf_like'], transcript_data, gene2tx, usage_df, reliability_df, sample_cols, epsilon=1e-6, n_cores=16)
    if sf_net is not None and ab.shape[0]>0:
        sf_net = pd.concat([ab, sf_net])


    # Concatenate TF like regulators if there are any
    ad = nn[nn.tfsf_category == 'tfsf_tf_like']
    tf_net = pd.concat([ad, tf_net])

    # the rest
    ambi_net = nn[nn.tfsf_category.isin(['tfsf_joint', 'tfsf_ambiguous'])]
else:
    ambi_net = None



if tf_net.shape[0]>0:
    tf_net = annotate_isoform_exclusive_edges(tf_net, tf_database, transcript_column='source_transcript')
    tf_net = annotate_isoform_exclusive_edges(tf_net, tf_database, transcript_column='target_transcript', suffixes = ('_source', '_target'))
    tf_net.to_csv(op.join(results_path, f'{experiment_name}.tf.tsv'), sep='\t')

if (ambi_net is not None) and (ambi_net.shape[0]>0):
    ambi_net = annotate_isoform_exclusive_edges(ambi_net, tf_database, transcript_column='source_transcript')
    ambi_net = annotate_isoform_exclusive_edges(ambi_net, tf_database, transcript_column='target_transcript', suffixes = ('_source', '_target'))
    ambi_net.to_csv(op.join(results_path, f'{experiment_name}.ambi.tsv'), sep='\t')

if (sf_net is not None) and sf_net.shape[0]>0:
    sf_net = annotate_isoform_exclusive_edges(sf_net, tf_database, transcript_column='source_transcript')
    sf_net = annotate_isoform_exclusive_edges(sf_net, tf_database, transcript_column='target_transcript', suffixes = ('_source', '_target'))
    sf_net.to_csv(op.join(results_path, f'{experiment_name}.sf.tsv'), sep='\t')

print(op.join(results_path, f'{experiment_name}.sf.tsv'))


